In [1]:
# ────────────────────────────────────────────────────────────────
# Copy & paste everything from here ↓
# ────────────────────────────────────────────────────────────────

import sys
import os
from pathlib import Path

# 1) Locate this script (or notebook) directory
try:
    script_dir = Path(__file__).resolve().parent
except NameError:
    # __file__ doesn't exist in notebooks or REPLs
    script_dir = Path.cwd()

# 2) Assume project root is one level up from `python/`
project_root = script_dir.parent

# 3) Sanity check: ensure there's a `data/` folder at the root
if not (project_root / "data").is_dir():
    raise RuntimeError(f"Project root {project_root!r} has no data/ folder.")

# 4) Prepend to sys.path so you can `import` anywhere in Music_Project
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Now you can freely do:
#   import pandas as pd
#   df = pd.read_csv(project_root / "data" / "clean" / "covers_clean.csv")
# ────────────────────────────────────────────────────────────────
# Copy & paste everything above ↑
# ────────────────────────────────────────────────────────────────

In [2]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import wikipediaapi
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures import ThreadPoolExecutor
import threading
import time
import ast
from collections import Counter

# Load data
df_orig = pd.read_csv(project_root / "data" / "raw" / "originals.csv")
df_cov = pd.read_csv(project_root / "data" / "raw" / "covers.csv")
df_neo = pd.read_csv("https://raw.githubusercontent.com/freiraum-bq/Music_Project/main/data/raw/neo4j_artists.csv")


# The Wikapedia Part of the Graph
requires: neo4j data import

next code cell is redundant (old version of wiki api and graph creation)

In [71]:
# Loading Wikapedia
wiki = wikipediaapi.Wikipedia(user_agent="MusicGraphExample (research@example.com)", language="en")

# Helpful Dataframes
df_neo_artists = df_neo[['artist_id', 'common_name', 'wiki_url']]
df_neo_artists = df_neo_artists[df_neo_artists['wiki_url'].notna() & (df_neo_artists['wiki_url'] != '')]

artist_urls = dict(zip(df_neo_artists['artist_id'], df_neo_artists['wiki_url']))
url_to_artist = {url: id for id, url in artist_urls.items()}
id_to_name = dict(zip(df_neo_artists['artist_id'], df_neo_artists['common_name']))

def get_page_links(url):
    tries = 3
    wiki = wikipediaapi.Wikipedia(user_agent="MusicGraphExample (research@example.com)", language="en")
    for attempt in range(tries):
        try:
            if '/wiki/' not in url:
                return url, set()
            page_title = url.split('/wiki/')[-1]
            page = wiki.page(page_title)
            if not page.exists():
                return url, set()
            links = page.links.keys()
            full_urls = {f"https://en.wikipedia.org/wiki/{link}" for link in links}
            return url, full_urls
        except Exception as e:
            if attempt == tries - 1:
                print(f"Exception in get_page_links for url {url}: {e}")
                return url, set()
            else:
                time.sleep(2 ** attempt)

def build_graph_threaded(artist_urls, url_to_artist, id_to_name, max_workers=10, batch_print=200):
    G = nx.DiGraph()
    artist_ids = list(artist_urls.keys())
    total = len(artist_ids)
    lock = threading.Lock()
    progress = {'count': 0}

    for artist in artist_ids:
        G.add_node(artist, name=id_to_name.get(artist, "Unknown"))

    def worker(artist):
        url = artist_urls[artist]
        url, linked_urls = get_page_links(url)
        edges = []
        for linked_url in linked_urls:
            if linked_url in url_to_artist and linked_url != url:
                mentioned_artist = url_to_artist[linked_url]
                edges.append((artist, mentioned_artist))
        with lock:
            progress['count'] += 1
            if progress['count'] % batch_print == 0 or progress['count'] == total:
                print(f"Processed {progress['count']} / {total} artists...")
        return edges

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(worker, artist_ids)

        for edges in results:
            for u, v in edges:
                G.add_edge(u, v, relation={'MENTIONS'})

    print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G

# Usage example:
G = build_graph_threaded(artist_urls, url_to_artist, id_to_name)

Processed 200 / 5828 artists...
Processed 400 / 5828 artists...
Processed 600 / 5828 artists...
Processed 800 / 5828 artists...
Processed 1000 / 5828 artists...
Processed 1200 / 5828 artists...
Processed 1400 / 5828 artists...
Processed 1600 / 5828 artists...
Processed 1800 / 5828 artists...
Processed 2000 / 5828 artists...
Processed 2200 / 5828 artists...
Processed 2400 / 5828 artists...
Processed 2600 / 5828 artists...
Processed 2800 / 5828 artists...
Processed 3000 / 5828 artists...
Processed 3200 / 5828 artists...
Processed 3400 / 5828 artists...
Processed 3600 / 5828 artists...
Processed 3800 / 5828 artists...
Processed 4000 / 5828 artists...
Processed 4200 / 5828 artists...
Processed 4400 / 5828 artists...
Processed 4600 / 5828 artists...
Processed 4800 / 5828 artists...
Processed 5000 / 5828 artists...
Processed 5200 / 5828 artists...
Processed 5400 / 5828 artists...
Processed 5600 / 5828 artists...
Processed 5800 / 5828 artists...
Processed 5828 / 5828 artists...
Graph built wi

next code cells is the new version which seperates the wiki loading and graph creation so the graph can be created fast

In [12]:
# Helpful Dataframes (assumed already loaded)
df_neo_artists = df_neo[['artist_id', 'common_name', 'wiki_url']]
df_neo_artists = df_neo_artists[df_neo_artists['wiki_url'].notna() & (df_neo_artists['wiki_url'] != '')]

artist_urls = dict(zip(df_neo_artists['artist_id'], df_neo_artists['wiki_url']))
url_to_artist = {url: id for id, url in artist_urls.items()}
id_to_name = dict(zip(df_neo['artist_id'], df_neo['common_name']))

In [ ]:
# Loading Wikipedia API once
wiki = wikipediaapi.Wikipedia(user_agent="MusicGraphExample (research@example.com)", language="en")

def get_all_links_for_artists(artist_urls, max_workers=10, batch_print=200):
    """
    Run the Wikipedia API calls once, store all linked URLs per artist.
    Returns: dict mapping artist_id -> set of linked artist URLs
    """
    artist_ids = list(artist_urls.keys())
    total = len(artist_ids)
    lock = threading.Lock()
    progress = {'count': 0}
    all_links = {}

    def worker(artist):
        url = artist_urls[artist]
        tries = 3
        for attempt in range(tries):
            try:
                if '/wiki/' not in url:
                    return artist, set()
                page_title = url.split('/wiki/')[-1]
                page = wiki.page(page_title)
                if not page.exists():
                    return artist, set()
                links = page.links.keys()
                full_urls = {f"https://en.wikipedia.org/wiki/{link}" for link in links}
                return artist, full_urls
            except Exception as e:
                if attempt == tries - 1:
                    print(f"Exception in get_page_links for url {url}: {e}")
                    return artist, set()
                else:
                    time.sleep(2 ** attempt)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(worker, artist_ids)

        for artist, linked_urls in results:
            with lock:
                progress['count'] += 1
                if progress['count'] % batch_print == 0 or progress['count'] == total:
                    print(f"Processed {progress['count']} / {total} artists...")
            all_links[artist] = linked_urls

    return all_links


all_links = get_all_links_for_artists(artist_urls)

Processed 200 / 5828 artists...
Processed 400 / 5828 artists...
Processed 600 / 5828 artists...
Processed 800 / 5828 artists...
Processed 1000 / 5828 artists...
Processed 1200 / 5828 artists...
Processed 1400 / 5828 artists...
Processed 1600 / 5828 artists...
Processed 1800 / 5828 artists...
Processed 2000 / 5828 artists...
Processed 2200 / 5828 artists...
Processed 2400 / 5828 artists...
Processed 2600 / 5828 artists...
Processed 2800 / 5828 artists...
Processed 3000 / 5828 artists...
Processed 3200 / 5828 artists...
Processed 3400 / 5828 artists...
Processed 3600 / 5828 artists...
Processed 3800 / 5828 artists...
Processed 4000 / 5828 artists...
Processed 4200 / 5828 artists...
Processed 4400 / 5828 artists...
Processed 4600 / 5828 artists...
Processed 4800 / 5828 artists...
Processed 5000 / 5828 artists...
Processed 5200 / 5828 artists...
Processed 5400 / 5828 artists...
Processed 5600 / 5828 artists...
Processed 5800 / 5828 artists...
Processed 5828 / 5828 artists...
Graph built wi

In [13]:
id_to_name

{1: 'Sidney Bechet',
 5: 'Take That',
 6: 'Van Morrison',
 7: 'Clouseau',
 8: 'Dinah Washington',
 10: 'Aretha Franklin',
 11: 'Marie Fredriksson',
 12: 'Nydia Rojas',
 14: 'The Beach Boys',
 16: 'The Doors',
 17: 'Led Zeppelin',
 20: 'Counting Crows',
 21: 'Chet Atkins',
 22: 'Sleepy John Estes',
 23: 'Taj Mahal',
 25: 'Janez Detd.',
 26: 'Alphaville',
 28: 'Missing Persons',
 29: 'Nina Simone',
 30: 'The Animals',
 31: "Screamin' Jay Hawkins",
 33: 'Bee Gees',
 34: 'Pete Seeger',
 35: 'The Byrds',
 37: 'Atomic Kitten',
 38: 'Steps',
 40: 'Barry Manilow',
 41: 'The Beatles',
 42: 'Joe Cocker',
 43: 'Merry Clayton',
 44: 'Betty Everett',
 45: 'Cher',
 47: 'Westlife',
 48: 'Extreme',
 49: 'Blondie',
 50: 'Sleeper',
 51: 'The Paragons',
 52: 'The Smashing Pumpkins',
 53: 'The Cars',
 54: 'The Cure',
 56: 'Letters to Cleo',
 57: 'Cyndi Lauper',
 58: 'Sheb Wooley',
 59: 'The Chipmunks',
 60: 'Patrice Rushen',
 61: 'Will Smith',
 63: 'The Smiths',
 64: 'Love Spit Love',
 65: 'Arrows',
 66: 

In [14]:
def build_graph_from_links(all_links, url_to_artist, id_to_name):
    """
    Build the graph using pre-fetched all_links dictionary.
    """
    G = nx.DiGraph()
    for artist in all_links.keys():
        G.add_node(artist, name=id_to_name.get(artist, "Unknown"))

    for artist, linked_urls in all_links.items():
        for linked_url in linked_urls:
            if linked_url in url_to_artist and linked_url != artist_urls[artist]:
                mentioned_artist = url_to_artist[linked_url]
                G.add_edge(artist, mentioned_artist, relation={'MENTIONS'})

    print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G


G = build_graph_from_links(all_links, url_to_artist, id_to_name)

Graph built with 5828 nodes and 10084 edges.


In [15]:
# Basic stats
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Count mentions: who is mentioned most (incoming edges)
mention_count = Counter(v for _, v in G.edges())
top10 = mention_count.most_common(10)

print("Top-10 most-mentioned artists:")
for name, cnt in top10:
    print(f"{G.nodes[name]['name']}: mentioned by {cnt} pages")

Nodes: 5828
Edges: 10084
Top-10 most-mentioned artists:
Adele: mentioned by 389 pages
Beyoncé: mentioned by 373 pages
U2: mentioned by 358 pages
Eminem: mentioned by 316 pages
Madonna: mentioned by 312 pages
Metallica: mentioned by 255 pages
Aerosmith: mentioned by 242 pages
Coldplay: mentioned by 242 pages
Rihanna: mentioned by 240 pages
Bono: mentioned by 225 pages


# Me thinking - the genre inclusion
A practice example is included below for future use

In [5]:
import pandas as pd

# 1) Read in your full artist CSV
path = project_root/"data/scraping/genre/artist_genres.csv"
df_genres = pd.read_csv(path)

# 2) Collect the genre columns into a list, dropping empties/NaNs
genre_cols = [f"genre_{i}" for i in range(1, 14)]
df_genres["genres"] = (
    df_genres[genre_cols]
      .apply(lambda row: [g for g in row if pd.notna(g) and g != ""], axis=1)
)

# 3) Index by the same artist key you use in G (common_name)
df_genres.set_index("artist_id", inplace=True)

# 4) Attach the genres list to each node in G
for artist in G.nodes():
    if artist in df_genres.index:
        G.nodes[artist]["genres"] = df_genres.at[artist, "genres"]
    else:
        G.nodes[artist]["genres"] = []

# 5) Verify on a few nodes
for artist in list(G.nodes())[:5]:
    print(G.nodes[artist]['name'], "→", G.nodes[artist]["genres"])

Van Morrison → ['r&b & soul', 'world', 'rock', 'electronic', 'folk', 'blues / country', 'jazz', 'other', 'pop']
Dinah Washington → ['jazz', 'blues / country', 'electronic', 'other', 'pop']
Aretha Franklin → ['r&b & soul', 'electronic', 'other', 'jazz', 'pop']
Marie Fredriksson → ['rock']
The Beach Boys → ['rock', 'pop']


In [58]:
rock_edges_both = [
    (u, v)
    for u, v in G.edges()
    if 'rock' in G.nodes[u].get('genres', []) and
       'rock' in G.nodes[v].get('genres', [])
]

rock_nodes_both = {n for edge in rock_edges_both for n in edge}

print(f"Number of nodes in 'rock ↔ rock' edges: {len(rock_nodes_both)}")
print("Nodes:", rock_nodes_both)

Number of nodes in 'rock ↔ rock' edges: 1317
Nodes: {24576, 6, 65543, 11, 14, 16, 17, 16401, 20, 28, 24608, 33, 24609, 35, 24611, 8229, 41, 42, 45, 48, 49, 32819, 52, 53, 54, 57, 57409, 69, 73, 75, 76, 32847, 8273, 8274, 41044, 85, 88, 90, 24666, 96, 100, 101, 32870, 103, 65639, 105, 24683, 116, 119, 128, 134, 8337, 150, 152, 155, 156, 24733, 158, 162, 167, 169, 32937, 171, 16569, 187, 191, 193, 198, 201, 204, 205, 211, 212, 73941, 214, 223, 41184, 41185, 227, 229, 230, 233, 33002, 240, 49400, 41208, 8445, 98562, 259, 260, 269, 281, 282, 24857, 8476, 24858, 65818, 291, 293, 294, 297, 298, 8489, 8491, 8492, 8493, 303, 306, 307, 24885, 98628, 326, 334, 65872, 347, 354, 361, 372, 374, 375, 49527, 377, 378, 8569, 382, 387, 391, 394, 8587, 402, 49565, 16802, 33187, 423, 8636, 33216, 451, 41415, 457, 41424, 16852, 8661, 33240, 41443, 484, 8690, 512, 74241, 90624, 515, 524, 526, 527, 74258, 533, 535, 25112, 537, 25113, 543, 545, 556, 557, 559, 25147, 8764, 8765, 584, 33358, 33359, 597, 600, 6

# The Covered Relation Addition

In [16]:
# Merge on org_perf_id column in df_cov and perf_id in df_orig, keep only cov_art_id and org_art_id 
# Not merging on song title because songs can have the same names
df_merged = pd.merge(
    df_cov[['org_perf_id', 'cov_art_id']],
    df_orig[['perf_id', 'org_art_id']],
    left_on='org_perf_id',
    right_on='perf_id',
    how='inner'
)[['cov_art_id', 'org_art_id']]

# org_perf_id and perf_id
print(df_merged.head())
print(df_merged.shape)

# # Merge on org_perf_id column in df_cov and perf_id in df_orig, keep only cov_art_id and org_art_id 
# # Not merging on song title because songs can have the same names
# df_merged_2 = pd.merge(df_cov[['song_title', 'cov_art_id']], 
#                      df_orig[['song_title', 'org_art_id']], 
#                      on='song_title', 
#                      how='inner')

# # org_perf_id and perf_id
# print(df_merged.head())
# print(df_merged.shape)
# print(df_merged_2.shape)
# print(df_orig['song_title'].duplicated().sum())
# print(df_orig['perf_id'].duplicated().sum())
# print(df_cov['perf_id'].duplicated().sum())

  cov_art_id org_art_id
0      [879]        [1]
1   [5, 237]     [5483]
2        [7]        [6]
3       [10]        [8]
4        [9]        [8]
(367622, 2)


In [17]:
# Step 1: Parse stringified lists if needed
def parse_list(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return val

df_merged['cov_art_id'] = df_merged['cov_art_id'].apply(parse_list)
df_merged['org_art_id'] = df_merged['org_art_id'].apply(parse_list)

# Step 2: Explode both columns
df_exploded = df_merged.explode('cov_art_id').explode('org_art_id').reset_index(drop=True)
print(df_exploded)

       cov_art_id org_art_id
0             879          1
1               5       5483
2             237       5483
3               7          6
4              10          8
...           ...        ...
482896      64925      11276
482897      35611      17062
482898      14339       2437
482899      14339      69125
482900        894       1389

[482901 rows x 2 columns]


In [18]:
edge_count = 0
for _, row in df_merged.iterrows():
    covering_artists = row['cov_art_id']
    original_artists = row['org_art_id']

    # Create edges for every pair (covering -> original)
    for cov_id in covering_artists:
        for org_id in original_artists:
            if cov_id != org_id: # avoid self-loop #and cov_id in G and org_id in G:
                
                # Ensure nodes exist with name attributes
                if cov_id not in G:
                    G.add_node(cov_id, name=id_to_name.get(cov_id, "Unknown"))
                if org_id not in G:
                    G.add_node(org_id, name=id_to_name.get(org_id, "Unknown"))

                if G.has_edge(cov_id, org_id):
                    edge_data = G[cov_id][org_id]
                    edge_data.setdefault('relation', set()).add('COVERED')
                    edge_data['weight'] = edge_data.get('weight', 0) + 1
                else:
                    G.add_edge(cov_id, org_id, relation={'COVERED'}, weight=1)
                edge_count += 1

In [25]:
import collections

# Step 2: Count how many times each artist is covered
cover_count = collections.Counter()
print(cover_count)
for u, v, data in G.edges(data=True):
    if 'COVERED' in data.get('relation', []):
        cover_count[v] += 1  # v = original artist who was covered

# Step 3: Get top 10 most covered artists
top_10_covered = cover_count.most_common(10)
print(top_10_covered)

# Step 4: Create the DataFrame
covered_top_10 = pd.DataFrame(
    [(artist, G.nodes[artist]['name'], count) for artist, count in top_10_covered],
    columns=['artist_id', 'artist_name', 'cover_count']
)

covered_top_10

Counter()
[(41, 4448), (4305, 2450), (243, 2413), (158, 1908), (2232, 1841), (319, 1654), (2223, 1590), (1424, 1550), (5773, 1437), (5774, 1437)]


,artist_id,artist_name,cover_count
0,41,The Beatles,4448
1,4305,Duke Ellington,2450
2,243,Bing Crosby,2413
3,158,Bob Dylan,1908
4,2232,Fred Astaire,1841
5,319,Frank Sinatra,1654
6,2223,Abbie Mitchell,1590
7,1424,Judy Garland,1550
8,5773,Joseph Mohr,1437
9,5774,Franz Gruber,1437


In [21]:
print(f"Total nodes: {len(G.nodes)}")

# Find artist IDs/names that shouldn't coexist
mixed_nodes = [n for n in G.nodes if isinstance(n, int)]  # IDs
named_nodes = [n for n in G.nodes if isinstance(n, str)]  # Names

print(f"Nodes with IDs: {len(mixed_nodes)}")
print(f"Nodes with names: {len(named_nodes)}")

Total nodes: 65833
Nodes with IDs: 65833
Nodes with names: 0


# Metrics

pagerank centrality metric
importance based on network structure

In [26]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 65833
Edges: 414104


### Overall metrics (lumpsumming covered and mentions edges)

In [31]:
# Compute PageRank
pagerank_scores = nx.pagerank(G, alpha=0.85) # is a default
top_10_pagerank = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:10]

# Create DataFrame
pagerank_top_10 = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top_10_pagerank],
    'artist_name': [G.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top_10_pagerank],
    'pagerank_score': [score for _, score in top_10_pagerank]
})

pagerank_top_10

,artist_id,artist_name,pagerank_score
0,41,The Beatles,0.011602
1,158,Bob Dylan,0.005468
2,297,U2,0.004613
3,84,Madonna,0.003464
4,616,Depeche Mode,0.003269
5,243,Bing Crosby,0.003229
6,206,The Rolling Stones,0.003206
7,10590,Coldplay,0.003054
8,8358,Beyoncé,0.003043
9,327,Chuck Berry,0.002990


In-degree Centrality

In [46]:
# Compute in-degree centrality (for directed graphs)
in_degree_centrality = nx.in_degree_centrality(G)

# Sort and get top 10 nodes by in-degree centrality
in_degree_centrality_results = sorted(in_degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]

# Create DataFrame
in_degree_top_10 = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in in_degree_centrality_results],
    'artist_name': [G.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in in_degree_centrality_results],
    'in_degree': [score for _, score in in_degree_centrality_results]
})

in_degree_top_10

,artist_id,artist_name,in_degree
0,41,The Beatles,0.067566
1,4305,Duke Ellington,0.037216
2,243,Bing Crosby,0.036654
3,158,Bob Dylan,0.028983
4,2232,Fred Astaire,0.027965
5,319,Frank Sinatra,0.025125
6,2223,Abbie Mitchell,0.024152
7,1424,Judy Garland,0.023545
8,5773,Joseph Mohr,0.021828
9,5774,Franz Gruber,0.021828


Betweenness Centrality

Nodes that control flow / connectors

Example in Your Music Network Context
Betweenness could highlight artists who link different genres or scenes by covering songs from multiple communities.

Artists who bridge otherwise disconnected groups or influence multiple clusters.

They might not be the most covered (high in-degree) but have strategic importance connecting parts of the network.



In [34]:
# Approximate betweenness using a sample of nodes
# note: run time is insane otherwise (could take days), thus k set low
betweenness = nx.betweenness_centrality(G, k=100)  # try k=500 or lower

# Sort nodes by betweenness centrality descending and take top 10
top_10_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]

# Create DataFrame
btw_degree_top_10 = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top_10_betweenness],
    'artist_name': [G.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top_10_betweenness],
    'btw_degree': [score for _, score in top_10_betweenness]
})

print(btw_degree_top_10)

   artist_id    artist_name  btw_degree
0         41    The Beatles    0.010200
1        158      Bob Dylan    0.008991
2         45           Cher    0.006350
3        319  Frank Sinatra    0.005720
4        103  Elvis Presley    0.005652
5        495   Petula Clark    0.004538
6        243    Bing Crosby    0.004345
7        754    Ray Charles    0.003831
8       1085    Percy Faith    0.003175
9         81   Jacques Brel    0.003038


### Metric on COVERED edges

In [53]:
# Covered-only subgraph (with weights)
edges_covered = [(u, v) for u, v, d in G.edges(data=True) if 'COVERED' in d.get('relation', set())]
G_covered = G.edge_subgraph(edges_covered).copy()

In [54]:
u, v, data = max(G_covered.edges(data=True), key=lambda x: x[2].get('weight', 0))
print(f"Edge with highest weight: {u} → {v}")
print("  Names:", G_covered.nodes[u].get('name', 'Unknown'), "→", G_covered.nodes[v].get('name', 'Unknown'))
print("  Relation:", data.get('relation'))
print("  Weight:", data.get('weight'))


Edge with highest weight: 97325 → 41
  Names: The Coverbeats → The Beatles
  Relation: {'COVERED'}
  Weight: 135


In [55]:
# Compute PageRank on the 'COVERED' subgraph using weights
pagerank_scores = nx.pagerank(G_covered, alpha=0.85, weight='weight')
top_10_pagerank = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:10]

# Create DataFrame
pagerank_top_10 = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top_10_pagerank],
    'artist_name': [G_covered.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top_10_pagerank],
    'pagerank_score': [score for _, score in top_10_pagerank]
})

pagerank_top_10

,artist_id,artist_name,pagerank_score
0,41,The Beatles,0.012941
1,158,Bob Dylan,0.005808
2,243,Bing Crosby,0.003622
3,616,Depeche Mode,0.003433
4,327,Chuck Berry,0.003406
5,4305,Duke Ellington,0.003387
6,206,The Rolling Stones,0.003340
7,26565,Prince's Band,0.003082
8,90,The Velvet Underground,0.002862
9,1886,Hank Williams,0.002777


In [59]:
# In-degree centrality for COVERED
in_deg_cov = nx.in_degree_centrality(G_covered)
top10_cov = sorted(in_deg_cov.items(), key=lambda x: x[1], reverse=True)[:10]
in_degree_cov_top10 = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top10_cov],
    'artist_name': [G.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top10_cov],
    'in_degree_centrality': [score for _, score in top10_cov]
})
in_degree_cov_top10

,artist_id,artist_name,in_degree_centrality
0,41,The Beatles,0.069111
1,4305,Duke Ellington,0.038067
2,243,Bing Crosby,0.037492
3,158,Bob Dylan,0.029646
4,2232,Fred Astaire,0.028605
5,319,Frank Sinatra,0.025699
6,2223,Abbie Mitchell,0.024705
7,1424,Judy Garland,0.024083
8,5773,Joseph Mohr,0.022328
9,5774,Franz Gruber,0.022328


In [61]:
# Covered betweenness
btw_covered = nx.betweenness_centrality(G_covered, k=100)
top10_btw_cov = sorted(btw_covered.items(), key=lambda x: x[1], reverse=True)[:10]

btw_cov_df = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top10_btw_cov],
    'artist_name': [G_covered.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top10_btw_cov],
    'btw_degree': [score for _, score in top10_btw_cov]
})

btw_cov_df

,artist_id,artist_name,btw_degree
0,41,The Beatles,0.005838
1,103,Elvis Presley,0.005821
2,158,Bob Dylan,0.005595
3,243,Bing Crosby,0.005371
4,319,Frank Sinatra,0.005363
5,2193,Johnny Mathis,0.004401
6,665,Johnny Cash,0.004313
7,206,The Rolling Stones,0.004295
8,84,Madonna,0.004295
9,754,Ray Charles,0.003752


### Metrics on MENTIONS edges

In [56]:
# Create a subgraph using only 'MENTIONS' edges (ignoring weight)
edges_mentions = [(u, v) for u, v, d in G.edges(data=True) if 'MENTIONS' in d.get('relation', set())]
G_mentions = G.edge_subgraph(edges_mentions).copy()

In [57]:
# Compute PageRank on 'MENTIONS' subgraph
pagerank_mentions = nx.pagerank(G_mentions, alpha=0.85)

# Get top 10
top_10_mentions = sorted(pagerank_mentions.items(), key=lambda x: x[1], reverse=True)[:10]

# Create DataFrame
pagerank_mentions_top_10 = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top_10_mentions],
    'artist_name': [G.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top_10_mentions],
    'pagerank_score': [score for _, score in top_10_mentions]
})

pagerank_mentions_top_10

,artist_id,artist_name,pagerank_score
0,297,U2,0.027021
1,84,Madonna,0.026064
2,224,Eminem,0.025635
3,8358,Beyoncé,0.025471
4,10590,Coldplay,0.024237
5,223,Aerosmith,0.023370
6,211,Metallica,0.023003
7,19179,Rihanna,0.021258
8,30607,Adele,0.021227
9,248,Jay-Z,0.020720


In [60]:
# In-degree centrality for MENTIONS
in_deg_ment = nx.in_degree_centrality(G_mentions)
top10_ment = sorted(in_deg_ment.items(), key=lambda x: x[1], reverse=True)[:10]
in_degree_mentions_top10 = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top10_ment],
    'artist_name': [G.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top10_ment],
    'in_degree_centrality': [score for _, score in top10_ment]
})
in_degree_mentions_top10

,artist_id,artist_name,in_degree_centrality
0,30607,Adele,0.138533
1,8358,Beyoncé,0.132835
2,297,U2,0.127493
3,224,Eminem,0.112536
4,84,Madonna,0.111111
5,211,Metallica,0.090812
6,223,Aerosmith,0.086182
7,10590,Coldplay,0.086182
8,19179,Rihanna,0.085470
9,1060,Bono,0.080128


In [62]:
# Mentions betweenness
btw_mentions = nx.betweenness_centrality(G_mentions, k=100)
top10_btw_ment = sorted(btw_mentions.items(), key=lambda x: x[1], reverse=True)[:10]

btw_ment_df = pd.DataFrame({
    'artist_id': [artist_id for artist_id, _ in top10_btw_ment],
    'artist_name': [G_mentions.nodes[artist_id].get('name', 'Unknown') for artist_id, _ in top10_btw_ment],
    'btw_degree': [score for _, score in top10_btw_ment]
})
btw_ment_df

,artist_id,artist_name,btw_degree
0,84,Madonna,0.007023
1,297,U2,0.006963
2,14508,Korn,0.006905
3,211,Metallica,0.006440
4,8358,Beyoncé,0.005283
5,10590,Coldplay,0.005024
6,224,Eminem,0.004566
7,223,Aerosmith,0.004547
8,543,Soundgarden,0.004199
9,13053,Björk,0.004180


# Network Description
The total network

In [21]:
# Total Network
# Basic stats
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
density = nx.density(G)

# Components (weakly connected components for directed graph)
num_components = nx.number_weakly_connected_components(G)

# Average shortest path length (only valid on strongly connected graphs or components)
# We'll use the largest weakly connected component
largest_cc = max(nx.weakly_connected_components(G), key=len)
G_sub = G.subgraph(largest_cc)
try:
    avg_shortest_path = nx.average_shortest_path_length(G_sub)
except:
    avg_shortest_path = "N/A (graph not connected)"

# Create a table
metrics = {
    "Metric": [
        "Number of vertices (artists)",
        "Number of edges (song covers)",
        "Number of components",
        "Average shortest path length",
        "Density"
    ],
    "Value": [
        num_nodes,
        num_edges,
        num_components,
        avg_shortest_path,
        density
    ]
}

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

                       Metric                     Value
 Number of vertices (artists)                     70189
Number of edges (song covers)                    414401
         Number of components                      3220
 Average shortest path length N/A (graph not connected)
                      Density                  0.000084


The covered relation

In [22]:
# The Network: covered relation
# Basic stats
covered_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('relation') == 'COVERED']
# Extract artist IDs only (first value in each tuple)
num_covered_edges = len([tup[0] for tup in covered_edges])

# Get unique nodes involved in these edges
covered_nodes = len(set([u for u, v in covered_edges] + [v for u, v in covered_edges]))


# Compute density
G_covered = G.edge_subgraph(covered_edges).copy()
density_covered = nx.density(G_covered)

# Components (weakly connected components for directed graph)
num_components_covered = nx.number_weakly_connected_components(G_covered)

# Average shortest path length (only valid on strongly connected graphs or components)
# We'll use the largest weakly connected component
largest_cc_covered = max(nx.weakly_connected_components(G_covered), key=len)
G_sub_covered = G.subgraph(largest_cc_covered)
try:
    avg_shortest_path_covered = nx.average_shortest_path_length(G_sub_covered)
except:
    avg_shortest_path_covered = "N/A (graph not connected)"

# Create a table
metrics_covered = {
    "Metric": [
        "Number of vertices (artists)",
        "Number of edges (song covers)",
        "Number of components",
        "Average shortest path length",
        "Density"
    ],
    "Value": [
        covered_nodes,
        num_covered_edges,
        num_components_covered,
        avg_shortest_path_covered,
        density_covered
    ]
}

metrics_df_covered = pd.DataFrame(metrics_covered)
print(metrics_df_covered.to_string(index=False))

                       Metric                     Value
 Number of vertices (artists)                     64361
Number of edges (song covers)                    404317
         Number of components                       189
 Average shortest path length N/A (graph not connected)
                      Density                  0.000098


The Network: wiki relation

In [23]:
# The Network: wiki relation
# Basic stats
mentions_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('relation') == 'MENTIONS']
# Extract artist IDs only (first value in each tuple)
num_mentions_edges = len([tup[0] for tup in mentions_edges])

# Get unique nodes involved in these edges
mentions_nodes = len(set([u for u, v in mentions_edges] + [v for u, v in mentions_edges]))


# Compute density
G_mentions = G.edge_subgraph(mentions_edges).copy()
density_mentions = nx.density(G_mentions)

# Components (weakly connected components for directed graph)
num_components_mentions = nx.number_weakly_connected_components(G_mentions)

# Average shortest path length (only valid on strongly connected graphs or components)
# We'll use the largest weakly connected component
largest_cc_mentions = max(nx.weakly_connected_components(G_mentions), key=len)
G_sub_mentions = G.subgraph(largest_cc_mentions)
try:
    avg_shortest_path_mentions = nx.average_shortest_path_length(G_sub_mentions)
except:
    avg_shortest_path_mentions = "N/A (graph not connected)"

# Create a table
metrics_mentions = {
    "Metric": [
        "Number of vertices (artists)",
        "Number of edges (song covers)",
        "Number of components",
        "Average shortest path length",
        "Density"
    ],
    "Value": [
        mentions_nodes,
        num_mentions_edges,
        num_components_mentions,
        avg_shortest_path_mentions,
        density_covered
    ]
}

metrics_df_mentions = pd.DataFrame(metrics_mentions)
print(metrics_df_mentions.to_string(index=False))

                       Metric                     Value
 Number of vertices (artists)                      2809
Number of edges (song covers)                     10084
         Number of components                        12
 Average shortest path length N/A (graph not connected)
                      Density                  0.000098
